## Hansard data


In [1]:
from discovery_utils.synthesis.policy import policy_update
from discovery_utils.utils import google
import pandas as pd

from src import utils as src_utils
from src import PROJECT_DIR, logging

PROJECT_NAME = src_utils.PROJECT_NAME
OUTPUT_DIR = src_utils.OUTPUT_DIR / "mission_radar"

from discovery_utils.utils import (
    analysis_gtr,
    analysis,
    charts,
    google,
    google_slides,
)

import re

2025-04-16 09:29:36,917 - root - INFO - Using OpenAI
2025-04-16 09:29:37,085 - root - INFO - Using OpenAI
2025-04-16 09:29:37,345 - root - INFO - Using OpenAI


In [6]:

import datetime
# import datetime data type

def date_to_quarter(date: datetime.datetime) -> str:
    return f"{date.year}-Q{date.quarter}"

def get_present_quarter():
    today = datetime.date.today()
    return f"{today.year}-Q{int((today.month - 1) / 3) + 1}"

In [2]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    "district_heating",
    "energy_efficiency",
    "energy_grid", 
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
    "energy_storage",    
    # "renewables_general",    
    "solar",
    "wind"
    # "decarbonisation_general",    
]

In [3]:
missions = ["ASF"]

In [4]:
import datetime
def get_quarter_from_date(date:str) -> int:
    """Return the quarter number from a given YYYY-MM-DD date string."""
    _date = datetime.datetime.strptime(date, "%Y-%m-%d")
    return (_date.month-1)//3 + 1


def impute_missing_quarters(df, date_col="quarter", value_col="speeches", min_quarter=None, max_quarter=None):
    # Convert quarter strings to Period type for easy handling
    df[date_col] = pd.PeriodIndex(df[date_col], freq='Q')
    
    # Determine min and max quarters
    if min_quarter is None:
        min_quarter = df[date_col].min()
    if max_quarter is None:
        max_quarter = df[date_col].max()
    
    # Generate full range of quarters
    full_range = pd.period_range(start=min_quarter, end=max_quarter, freq='Q')
    
    # Create a complete DataFrame with all quarters
    full_df = pd.DataFrame({date_col: full_range})
    
    # Merge with the original DataFrame
    df = full_df.merge(df, on=date_col, how='left')
    
    # Fill missing values in # speeches column with 0
    df[value_col] = df[value_col].fillna(0).astype(int)
    
    # Convert period back to string if necessary
    df[date_col] = df[date_col].astype(str)
    # Add a dash between year and quarter
    df[date_col] = df[date_col].str.replace("Q", "-Q")
    return df

def impute_missing_years(df, year_col="year", value_col="speeches", min_year=None, max_year=None):
    # Ensure year column is integer
    df[year_col] = df[year_col].astype(int)

    # Determine min and max years
    if min_year is None:
        min_year = df[year_col].min()
    if max_year is None:
        max_year = df[year_col].max()

    # Generate a complete range of years
    full_range = pd.DataFrame({year_col: range(min_year, max_year + 1)})

    # Merge with the existing DataFrame
    df = full_range.merge(df, on=year_col, how='left')

    # Fill missing values in # speeches column with 0
    df[value_col] = df[value_col].fillna(0).astype(int)
    return df

In [5]:
HansardData = policy_update.HansardData()

2025-04-16 10:32:55,953 - discovery_utils.getters.hansard - INFO - Downloading debates parquet file: data/policy_scanning_data/enriched/HansardDebates.parquet
2025-04-16 10:33:49,675 - discovery_utils.getters.hansard - INFO - Attempting to download label store: data/policy_scanning_data/enriched/HansardDebates_LabelStore_keywords.csv
2025-04-16 10:34:59,814 - discovery_utils.getters.hansard - INFO - Downloading people metadata
2025-04-16 10:35:00,016 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-16 10:35:00,082 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-16 10:35:00,132 - botocore.httpchecksum - INFO - Skipping checksum validation. Response did not contain one of the following algorithms: ['crc32', 'sha1', 'sha256'].
2025-04-16 10:35:00,165 - botocore.httpchecksum

In [6]:
speeches_df = (
    HansardData.debates_df
    .query("date >= '2014-01-01' & date <= '2025-03-31'")
    .merge(
        HansardData.labelstore_df[['id', 'mission_labels', 'topic_labels']],
        left_on='speech_id',
        right_on='id',
        how='left'
    )
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))
    .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))    
    .explode("mission_labels")
    .query("mission_labels in @missions")
    .explode("topic_labels")
    .assign(quarter = lambda df: df.date.apply(get_quarter_from_date))
    .assign(quarter = lambda df: df.year.astype(str) + "-Q" + df.quarter.astype(str))
)

In [7]:
PRESENT_QUARTER = "2025-Q1"

In [8]:
def process_data(matching_ids, category_name: str, _speeches_df = speeches_df, prefix: str = None):

    present_quarter = PRESENT_QUARTER
    
    selected_df = (
        _speeches_df
        .query("speech_id in @matching_ids")
        .drop_duplicates(subset="speech_id")
        .assign(speech_text_norm = lambda df: df.speech.apply(lambda x: re.sub(r"\s+", " ", x)))
        .drop_duplicates(["speakername", "date", "speech_text_norm"])        
    )

    # Figure variables
    if prefix is None:
        prefix = f"{OUTPUT_DIR}/charts/hansard_{category_name}_"
    _scale = 2

    ts_quarterly_df = (
        selected_df
        .query("date >= '2023-01-01'")
        .groupby("quarter")
        .agg(speeches = ("speech_id", "count"))
        .reset_index()
        .pipe(impute_missing_quarters, min_quarter="2023Q1", max_quarter="2025Q1")
    )

    fig = charts.ts_bar(
        ts_quarterly_df,
        variable="speeches",
        variable_title="Number of speeches",
        time_column="quarter",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of speeches for {category_name}")
    chart_filename = f"{prefix}quarterly_speeches.png"
    fig.save(chart_filename, scale_factor=_scale)    

    
    previous_four_quarters = ts_quarterly_df.query("quarter < @present_quarter").sort_values("quarter").tail(4).quarter.tolist()
    previous_four_quarters_mean_df = (
        ts_quarterly_df
        .query("quarter in @previous_four_quarters")
        .assign(_col = "previous_four_quarters")
        .groupby("_col")
        .agg(
            amount=("speeches", "mean"),
        )
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    present_quarter_df = (
        ts_quarterly_df
        .query("quarter == @present_quarter")
        # rename index to "present_quarter"
        .assign(_col = "magnitude")
        .groupby("_col")
        .agg(
            amount=("speeches", "mean"),
        )
        .T.reset_index().rename(columns={"index": "variable"})
        )

    growth_magnitude_quarterly_df = (
        previous_four_quarters_mean_df
        .merge(present_quarter_df, on="variable", how="left")
        .assign(growth = lambda df: (df.magnitude - df.previous_four_quarters) / df.previous_four_quarters * 100)
        .assign(theme=category_name)
    )

    ts_df = (
        selected_df
        .query("date >= '2014-01-01'")
        .groupby("year")
        .agg(speeches = ("speech_id", "count"))
        .reset_index()
        .pipe(impute_missing_years, min_year=2014, max_year=2025)
    )

    fig = charts.ts_bar(
        ts_df,
        variable="speeches",
        variable_title="Number of speeches",
        time_column="year",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of speeches for {category_name}")
    chart_filename = f"{prefix}speeches.png"
    fig.save(chart_filename, scale_factor=_scale)        

    growth_magnitude_df = (
        analysis.magnitude_growth(ts_df, year_start=2020, year_end=2024)
        .assign(theme=category_name)
        .reset_index()
        .rename(columns={'index': 'variable'})
    )    

    return ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, selected_df

In [9]:
prefix = None

all_ts_df = []
all_ts_quarterly_df = []
all_growth_magnitude_df = []
all_growth_magnitude_quarterly_df = []
all_speeches_df = []

for config_name in CONFIG_NAMES:

    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]

    selected_df = (
        speeches_df
        .query(f"topic_labels == '{category_name}'")
        .drop_duplicates("speech_id")
    )        
    matching_ids = selected_df.speech_id.tolist()

    ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, selected_df = process_data(matching_ids, category_name, selected_df, prefix)

    all_ts_df.append(ts_df.assign(theme=category_name))
    all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
    all_growth_magnitude_df.append(growth_magnitude_df)
    all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
    all_speeches_df.append(selected_df.assign(theme=category_name))


In [10]:
low_carbon_heating_configs = [
    "biomass_heating",    
    "district_heating",
    "geothermal_energy",
    "heat_pumps",
    "heat_storage",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
]
lch_names = []
for config_name in low_carbon_heating_configs:
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    lch_names.append(category_name)

In [11]:
# Low-carbon heating
category_name = "Low-carbon heating"
selected_df = (
    speeches_df
    .query(f"topic_labels in @lch_names")
    .drop_duplicates("speech_id")
)        
matching_ids = selected_df.speech_id.tolist()

ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, selected_df = process_data(matching_ids, category_name, selected_df, prefix)

all_ts_df.append(ts_df.assign(theme=category_name))
all_ts_quarterly_df.append(ts_quarterly_df.assign(theme=category_name))
all_growth_magnitude_df.append(growth_magnitude_df)
all_growth_magnitude_quarterly_df.append(growth_magnitude_quarterly_df)
all_speeches_df.append(selected_df.assign(theme=category_name))

In [12]:
table_prefix = f"{OUTPUT_DIR}/hansard_"

In [13]:
all_ts_df = pd.concat(all_ts_df, ignore_index=True)
all_ts_df.to_csv(f"{table_prefix}all_ts_df.csv", index=False)

all_ts_quarterly_df = pd.concat(all_ts_quarterly_df, ignore_index=True)
all_ts_quarterly_df.to_csv(f"{table_prefix}all_ts_quarterly_df.csv", index=False)

all_growth_magnitude_df = pd.concat(all_growth_magnitude_df, ignore_index=True)
all_growth_magnitude_df.to_csv(f"{table_prefix}all_growth_magnitude_df.csv", index=False)

all_growth_magnitude_quarterly_df = pd.concat(all_growth_magnitude_quarterly_df, ignore_index=True)
all_growth_magnitude_quarterly_df.to_csv(f"{table_prefix}all_growth_magnitude_quarterly_df.csv", index=False)

all_speeches_df = pd.concat(all_speeches_df, ignore_index=True)
all_speeches_df.to_csv(f"{table_prefix}all_speeches_df.csv", index=False)

## Baseline

In [24]:
baseline_df = (
    HansardData.debates_df
    .query("date >= '2014-01-01' & date <= '2025-03-31'")
    .drop_duplicates(subset="speech_id")
    .assign(speech_text_norm = lambda df: df.speech.apply(lambda x: re.sub(r"\s+", " ", x)))
    .drop_duplicates(["speakername", "date", "speech_text_norm"])       
    .assign(quarter = lambda df: df.date.apply(get_quarter_from_date))
    .assign(quarter = lambda df: df.year.astype(str) + "-Q" + df.quarter.astype(str))
)

In [25]:
matching_ids = baseline_df.speech_id.to_list()
prefix=None
category_name = "All speeches"

In [26]:
ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, selected_df = process_data(matching_ids, category_name, _speeches_df=baseline_df, prefix=prefix)

In [29]:
ts_df

,year,speeches
0,2014,63606
1,2015,60289
2,2016,67365
3,2017,62609
4,2018,77659
5,2019,67653
6,2020,63142
7,2021,54927
8,2022,61494
9,2023,54775


In [28]:
growth_magnitude_df

,variable,magnitude,growth,theme
0,speeches,57456.8,-18.823817,All speeches


In [ ]:
growth_magnitude_quarterly_df

_col,variable,previous_four_quarters,magnitude,growth,theme
0,amount,13235.75,17503.0,32.240334,All speeches


## Specific analyses

In [ ]:
category_name = "Heat pumps"

selected_df = (
    speeches_df
    .query(f"topic_labels == '{category_name}'")
    .drop_duplicates("speech_id")
)   

matching_ids = selected_df.speech_id.tolist()

selected_df = (
    speeches_df
    .query(f"topic_labels == '{category_name}'")
    .query("speech_id in @matching_ids")
    .drop_duplicates(subset="speech_id")
    .assign(speech_text_norm = lambda df: df.speech.apply(lambda x: re.sub(r"\s+", " ", x)))
    .drop_duplicates(["speakername", "date", "speech_text_norm"])        
)

ts_quarterly_df = (
    selected_df
    .query("date >= '2021-01-01'")
    .groupby("quarter")
    .agg(speeches = ("speech_id", "count"))
    .reset_index()
    .pipe(impute_missing_quarters, min_quarter="2021Q1", max_quarter="2025Q1")
)

ts_quarterly_df

,quarter,speeches
0,2021-Q1,4
1,2021-Q2,7
2,2021-Q3,7
3,2021-Q4,39
4,2022-Q1,16
5,2022-Q2,11
6,2022-Q3,4
7,2022-Q4,5
8,2023-Q1,13
9,2023-Q2,22
